In [1]:
!pip install datasets transformers sentence-transformers faiss-cpu langchain evaluate rouge_score

  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.8/23.8 MB 115.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 84.1/84.1 kB 11.3 MB/s eta 0:00:00
  Created wheel for rouge_score: filename=rouge_score-0.1.2-py3-none-any.whl size=24934 sha256=1e1accbef082e14ff6773ab015676ccbd6f9cdb6a337a49da4eec9ff40724753
  Stored in directory: /root/.cache/pip/wheels/85/9d/af/01feefbe7d55ef5468796f0c68225b6788e85d9d0a281e7a70
Successfully built rouge_score


In [2]:
import time
import torch
import torch.nn as nn
import evaluate
from torch.utils.data import DataLoader

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
from datasets import load_dataset
from transformers import AutoTokenizer
#We decided to work with FiscalNote/billsum dataset from Hugging Face
dataset = load_dataset("FiscalNote/billsum")
#We decided to use pretrained Facebook/bart-base tokenizer which was pretrained on 50,265 tokens
tokenizer = AutoTokenizer.from_pretrained("facebook/bart-base")

README.md: 0.00B [00:00, ?B/s]

data/train-00000-of-00001.parquet:   0%|          | 0.00/91.8M [00:00<?, ?B/s]

data/test-00000-of-00001.parquet:   0%|          | 0.00/15.8M [00:00<?, ?B/s]

data/ca_test-00000-of-00001.parquet:   0%|          | 0.00/6.12M [00:00<?, ?B/s]

Generating train split:   0%|          | 0/18949 [00:00<?, ? examples/s]

...

Generating ca_test split:   0%|          | 0/1237 [00:00<?, ? examples/s]

config.json: 0.00B [00:00, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

In [3]:
print(dataset)

DatasetDict({
    train: Dataset({
        features: ['text', 'summary', 'title'],
        num_rows: 18949
    })
    test: Dataset({
        features: ['text', 'summary', 'title'],
        num_rows: 3269
    })
    ca_test: Dataset({
        features: ['text', 'summary', 'title'],
        num_rows: 1237
    })
})


## Model 3: Custom BART-Style Transformer Initialized with Pretrained BART Weights

Model 3 is designed to sit between the fully custom Transformer from Model 1 and the standard pretrained BART baseline from Model 2.

Instead of using the generic nn.Transformer, this model uses BART's encoder-decoder architecture and initializes from pretrained facebook/bart-base weights. To make it a custom BART-style variant, we wrap the pretrained BART architecture in our own class and add an extra dropout layer before the vocabulary projection during fine-tuning.

This lets us test a hybrid idea: pretrained BART knowledge plus a custom training path.

In [4]:
from transformers import BartForConditionalGeneration
from transformers.models.bart.modeling_bart import shift_tokens_right

class CustomBartStyleSummarizer(nn.Module):
  def __init__(self, model_name="facebook/bart-base", custom_dropout=0.2):
    super().__init__()

    #Load pretrained BART weights
    self.bart = BartForConditionalGeneration.from_pretrained(model_name)

    #extra dropout before the final vocab projection
    self.custom_dropout = nn.Dropout(custom_dropout)

  def forward(self, input_ids, attention_mask, labels=None):
    decoder_input_ids = None

    #labels are shifted right to create decoder inputs
    if labels is not None:
      decoder_input_ids = shift_tokens_right(labels,self.bart.config.pad_token_id,self.bart.config.decoder_start_token_id)

    # Run the pretrained BART encoder-decoder backbone
    outputs = self.bart.model(
        input_ids=input_ids,
        attention_mask=attention_mask,
        decoder_input_ids=decoder_input_ids,
        use_cache=False,
        return_dict=True
    )

    # Apply custom dropout to decoder hidden states
    decoder_hidden_states = self.custom_dropout(outputs.last_hidden_state)

    # Project decoder states to vocabulary logits
    logits = self.bart.lm_head(decoder_hidden_states) + self.bart.final_logits_bias

    loss = None
    if labels is not None:
      loss_fn = nn.CrossEntropyLoss(ignore_index=-100)
      loss = loss_fn(logits.reshape(-1, logits.size(-1)), labels.reshape(-1))

    return {"loss": loss, "logits": logits}

  def generate(self, *args, **kwargs):
    # Use BART's built-in generation method for inference
    return self.bart.generate(*args, **kwargs)


###Preprocessing

For BART-style models, padding tokens inside the target summaries should be replaced with -100. PyTorch's cross-entropy loss ignores labels with value -100, so the model is not penalized for predicting padding tokens.

Unlike Model 1, we do not manually create decoder inputs, causal masks, or decoder targets. BART handles decoder shifting and masking internally.

In [5]:
def preprocess_bart(data):
  source = tokenizer(data["text"], max_length=512, truncation=True, padding="max_length")
  target = tokenizer(data["summary"], max_length=128, truncation=True, padding="max_length")

  labels = target["input_ids"]
  new_labels = []
  for token in labels:
      if token != tokenizer.pad_token_id:
          new_labels.append(token)
      else:
          new_labels.append(-100)

  labels = new_labels

  source["labels"] = labels
  return source

In [6]:
# Use the same scale as the  custom Transformer experiment for a fair comparison
train_dataset_model3 = dataset["train"].select(range(5000))
val_dataset_model3 = dataset["test"].select(range(100))

tokenized_dataset_model3 = train_dataset_model3.map(preprocess_bart)
tokenized_dataset_model3.set_format(type="torch")

val_tokenized_model3 = val_dataset_model3.map(preprocess_bart)
val_tokenized_model3.set_format(type="torch")

dataloader_model3 = DataLoader(tokenized_dataset_model3, batch_size=4, shuffle=True)
val_dataloader_model3 = DataLoader(val_tokenized_model3, batch_size=4, shuffle=False)

Map:   0%|          | 0/5000 [00:00<?, ? examples/s]

Map:   0%|          | 0/100 [00:00<?, ? examples/s]

### Training

This training loop is simpler than the custom Transformer loop because BART internally handles the decoder input shift and causal masking. The only required inputs are input_ids, attention_mask, and labels.

In [7]:
torch.manual_seed(42)

model_3 = CustomBartStyleSummarizer(model_name="facebook/bart-base", custom_dropout=0.2)
model_3 = model_3.to(device)

optimizer_3 = torch.optim.AdamW(model_3.parameters(), lr=3e-5)

no_epochs_model3 = 3
best_vloss_model3 = 1_000_000.
start_time_model3 = time.time()

for epoch in range(no_epochs_model3):
    running_loss_model3 = 0.0
    epoch_start_time_model3 = time.time()

    model_3.train()
    for i, data in enumerate(dataloader_model3):
      input_ids = data["input_ids"].to(device)
      attention_mask = data["attention_mask"].to(device)
      labels = data["labels"].to(device)

      optimizer_3.zero_grad()

      # Forward pass  computes logits and loss using input_ids, attention_mask, and labels
      outputs = model_3(input_ids=input_ids, attention_mask=attention_mask, labels=labels)
      loss = outputs["loss"]
      loss.backward()
      torch.nn.utils.clip_grad_norm_(model_3.parameters(), max_norm=1.0)
      optimizer_3.step()

      running_loss_model3 += loss.item()

    model_3.eval()
    val_loss_model3 = 0.0
    with torch.no_grad():
      for i, val_data in enumerate(val_dataloader_model3):
        input_ids = val_data["input_ids"].to(device)
        attention_mask = val_data["attention_mask"].to(device)
        labels = val_data["labels"].to(device)

        outputs = model_3(
            input_ids=input_ids,
            attention_mask=attention_mask,
            labels=labels
        )

        val_loss_model3 += outputs["loss"].item()

    avg_train_loss_model3 = running_loss_model3 / len(dataloader_model3)
    avg_val_loss_model3 = val_loss_model3 / len(val_dataloader_model3)

    if avg_val_loss_model3 < best_vloss_model3:
      best_vloss_model3 = avg_val_loss_model3
      torch.save(model_3.state_dict(), "best_custom_bart_style_model3.pt")
      tokenizer.save_pretrained("model3_tokenizer")

    epoch_end_model3 = time.time()

    print( f"Epoch {epoch+1}/{no_epochs_model3} | ", f"Train Loss: {avg_train_loss_model3:.4f} | ", f"Val Loss: {avg_val_loss_model3:.4f} | ", f"Time: {epoch_end_model3 - epoch_start_time_model3:.2f}s")

end_time_model3 = time.time()
print("Finished Model 3 training. Time:", end_time_model3 - start_time_model3, "seconds")


model.safetensors:   0%|          | 0.00/558M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/259 [00:00<?, ?it/s]

Epoch 1/3 |  Train Loss: 2.5558 |  Val Loss: 1.8570 |  Time: 106.60s
Epoch 2/3 |  Train Loss: 2.1102 |  Val Loss: 1.7942 |  Time: 105.73s
Epoch 3/3 |  Train Loss: 1.9013 |  Val Loss: 1.7665 |  Time: 105.62s
Finished Model 3 training. Time: 317.94702553749084 seconds


### Generation and ROUGE Evaluation

For generation, Model 3 uses BART's generation method. Beam search and no-repeat n-gram blocking are included to reduce repetition and improve summary quality.

In [8]:
model_3.load_state_dict(torch.load("best_custom_bart_style_model3.pt", map_location=device))
model_3.eval()

sample_model3 = next(iter(val_dataloader_model3))

input_ids = sample_model3["input_ids"][0].unsqueeze(0).to(device)
attention_mask = sample_model3["attention_mask"][0].unsqueeze(0).to(device)

summary_ids = model_3.generate(
    input_ids=input_ids,
    attention_mask=attention_mask,
    max_length=128,
    num_beams=4,
    no_repeat_ngram_size=3,
    early_stopping=True
)

print("REFERENCE:")
ref_labels = sample_model3["labels"][0].clone()
ref_labels[ref_labels == -100] = tokenizer.pad_token_id
print(tokenizer.decode(ref_labels, skip_special_tokens=True))

print("MODEL 3 GENERATED:")
print(tokenizer.decode(summary_ids[0], skip_special_tokens=True))


REFERENCE:
Amends the Water Resources Development Act of 1999 to: (1) authorize appropriations for FY 1999 through 2009 for implementation of a long-term resource monitoring program with respect to the Upper Mississippi River Environmental Management Program (currently, such funding is designated for a program for the planning, construction, and evaluation of measures for fish and wildlife habitat rehabilitation and enhancement); (2) authorize the Secretary of the Army to carry out modifications to the navigation project for the Delaware River, Pennsylvania and Delaware, if such project as modified is technically sound, environmentally (currently, economically) acceptable, and economically justified; (3) subject certain previously deauthorized water
MODEL 3 GENERATED:
Amends the Water Resources Development Act of 1992 to direct the Secretary of the Interior, acting through the Director of the Environmental Protection Agency (EPA), to make grants to: (1) States for the elimination or co

In [9]:
rouge = evaluate.load("rouge")

predictions_model3 = []
references_model3 = []

num_samples_model3 = 30
count_model3 = 0

model_3.eval()

with torch.no_grad():
    for batch in val_dataloader_model3:
        input_ids_batch = batch["input_ids"].to(device)
        attention_mask_batch = batch["attention_mask"].to(device)
        labels_batch = batch["labels"]

        for i in range(len(input_ids_batch)):
            summary_ids = model_3.generate(
                input_ids=input_ids_batch[i].unsqueeze(0),
                attention_mask=attention_mask_batch[i].unsqueeze(0),
                max_length=128,
                num_beams=4,
                no_repeat_ngram_size=3,
                early_stopping=True
            )

            pred_text = tokenizer.decode(summary_ids[0], skip_special_tokens=True)

            ref_labels = labels_batch[i].clone()
            ref_labels[ref_labels == -100] = tokenizer.pad_token_id
            ref_text = tokenizer.decode(ref_labels, skip_special_tokens=True)

            predictions_model3.append(pred_text)
            references_model3.append(ref_text)

            count_model3 += 1
            if count_model3 >= num_samples_model3:
                break

        if count_model3 >= num_samples_model3:
            break

scores_model3 = rouge.compute(predictions=predictions_model3, references=references_model3)
print(scores_model3)


{'rouge1': np.float64(0.5068970039022869), 'rouge2': np.float64(0.31771544937938767), 'rougeL': np.float64(0.40335975368381227), 'rougeLsum': np.float64(0.4271855012347954)}


### Model 3 Analysis

Model 3 uses pretrained BART weights but changes the training path through a custom wrapper and dropout layer before the vocabulary projection. This should perform better than the fully custom Transformer trained from scratch because it starts with pretrained language knowledge, but it is still a distinct experimental model from standard BART fine-tuning.

### References
* For tokenization
https://huggingface.co/docs/transformers/tasks/summarization
* For Transformer Architecture
https://github.com/pytorch/examples/blob/main/language_translation/src/model.py